# 🎯 Notebook 6 — YOLOv8 Object Detection
## Aerial Bird vs Drone Detection

**Objectives:**
- Prepare dataset in YOLOv8 format
- Configure data.yaml
- Train YOLOv8 model
- Validate with mAP metrics
- Run inference on test images

In [ ]:
import os, sys
from pathlib import Path

ROOT = Path(os.getcwd()).parent
sys.path.insert(0, str(ROOT))

# Install ultralytics if needed
try:
    from ultralytics import YOLO
    print('✅ ultralytics installed')
except ImportError:
    print('Installing ultralytics...')
    os.system('pip install ultralytics -q')
    from ultralytics import YOLO

from yolov8.train_yolo import train, validate, run_inference, export_model
print('✅ YOLOv8 module loaded')

## 1. Prepare Detection Dataset

In [ ]:
# Generate YOLOv8-format labels from classified images
from scripts.generate_sample_labels import populate_detection_dataset
populate_detection_dataset()

## 2. Verify Dataset Structure

In [ ]:
detection_root = ROOT / 'data' / 'object_detection_dataset'

for split in ['train', 'val', 'test']:
    img_dir = detection_root / 'images' / split
    lbl_dir = detection_root / 'labels' / split
    n_imgs  = len(list(img_dir.glob('*.jpg'))) if img_dir.exists() else 0
    n_lbls  = len(list(lbl_dir.glob('*.txt'))) if lbl_dir.exists() else 0
    print(f'{split:<6}: images={n_imgs}  labels={n_lbls}')

# Show sample label content
sample_lbl = list((detection_root / 'labels' / 'train').glob('*.txt'))[0]
print(f'\nSample label ({sample_lbl.name}):')
print(sample_lbl.read_text())

## 3. Train YOLOv8

In [ ]:
# Training — adjust epochs for your hardware
model, results = train()

## 4. Validate

In [ ]:
metrics = validate()
if metrics:
    print(f'mAP@50   : {metrics.box.map50:.4f}')
    print(f'mAP@50-95: {metrics.box.map:.4f}')

## 5. Run Inference on Test Images

In [ ]:
test_img_dir = str(detection_root / 'images' / 'test')
results = run_inference(test_img_dir, conf=0.25, iou=0.45)

## 6. Export Model (Optional)

In [ ]:
# Export to ONNX for deployment
# exported = export_model(export_format='onnx')
print('To export: uncomment the export_model() call above')
print('Supported formats: onnx, tflite, coreml, torchscript')